# Tomahawk Rev2.4 integrated screening

This notebook runs the current Rev1.7 provisional inputs through a fixed-boundary axisymmetric Grad-Shafranov screen, D-T power/confinement balance, coolant conservation screen, and reduced uncertainty envelope. It runs a smoke case first and then the full case count. Results remain preliminary and do not claim 3-D MHD, OpenMC, CFD/CHT, CHF, ignition, or hardware validation.

In [ ]:
import importlib.util
import os
from pathlib import Path
import zipfile

OUT = Path(os.environ.get("TOMAHAWK_OUTPUT_ROOT", "/kaggle/working/tomahawk_rev24_screen")).resolve()
OUT.mkdir(parents=True, exist_ok=True)

roots = [Path.cwd()]
for text in ("/kaggle/input", "/kaggle/working"):
    path = Path(text)
    if path.exists():
        roots.append(path)

# If the package was uploaded as a ZIP dataset, extract only this package.
for root in list(roots):
    if not root.exists():
        continue
    for archive in root.rglob("Tomahawk_Rev2_4_Integrated_Screening_Package.zip"):
        target = Path("/kaggle/working/tomahawk_rev24_package")
        target.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive) as handle:
            handle.extractall(target)
        roots.append(target)

script_candidates = []
for root in roots:
    if root.exists():
        script_candidates.extend(root.rglob("Tomahawk_Rev2_4_Integrated_Screening.py"))
if not script_candidates:
    raise FileNotFoundError("Upload the Rev2.4 package ZIP or the .py file as a Kaggle input.")
SCRIPT = sorted(set(script_candidates), key=lambda p: ("/kaggle/input" not in str(p), len(str(p))))[0].resolve()
INPUT_ROOT = Path(os.environ.get("TOMAHAWK_INPUT_ROOT", str(SCRIPT.parent / "provisional_inputs"))).resolve()
if not INPUT_ROOT.is_dir():
    raise FileNotFoundError(f"Input root not found: {INPUT_ROOT}")
spec = importlib.util.spec_from_file_location("tomahawk_rev24_screen", SCRIPT)
screen = importlib.util.module_from_spec(spec)
spec.loader.exec_module(screen)
print("Script:", SCRIPT)
print("Inputs:", INPUT_ROOT)
print("Output:", OUT)

## Smoke run

The smoke run checks imports, profile closure, the sparse equilibrium solve, the coolant-row parser, strict JSON writing, and the explicit blocker policy.

In [ ]:
smoke = screen.run_native_screen(
    input_root=INPUT_ROOT,
    output_root=OUT / "smoke",
    label="smoke",
    cases=int(os.environ.get("TOMAHAWK_SMOKE_CASES", "256")),
    seed=20260905,
    nr=161,
    nz=161,
    max_gs_iters=12000,
)
print({
    "overall_status": smoke["overall_status"],
    "GS_residual_pass": smoke["gates"]["GS_residual_pass"],
    "profile_consistency_pass": smoke["gates"]["profile_consistency_pass"],
    "confinement_feasibility_screen_pass": smoke["gates"]["confinement_feasibility_screen_pass"],
})

## Full run

The full run repeats the auditable solve and evaluates the reduced 65,536-case parameter envelope. The uncertainty cases are scaled screens and do not re-solve Grad-Shafranov for every sample.

In [ ]:
full = screen.run_native_screen(
    input_root=INPUT_ROOT,
    output_root=OUT / "full",
    label="full",
    cases=int(os.environ.get("TOMAHAWK_FULL_CASES", "65536")),
    seed=20260906,
    nr=161,
    nz=161,
    max_gs_iters=12000,
)
print({
    "overall_status": full["overall_status"],
    "confinement_numerical_screen_pass": full["gates"]["confinement_numerical_screen_pass"],
    "fusion_GW_module": full["confinement_metrics"]["fusion_GW_module"],
    "alpha_self_heating_ratio": full["confinement_metrics"]["alpha_self_heating_ratio"],
    "reference_condition_self_heating_ratio": full["confinement_metrics"]["reference_condition_self_heating_ratio"],
    "reference_self_heating_reconciliation_pass": full["gates"]["reference_self_heating_reconciliation_pass"],
    "coolant_screen_pass": full["gates"]["coolant_screen_pass"],
    "all_gate_pass_percent": full["uncertainty_summary"]["all_gate_pass_percent"],
})

In [ ]:
import json
for name in ("equilibrium_metrics.json", "confinement_metrics.json", "coolant_summary.json", "gate_results.json"):
    path = OUT / "full" / name
    print("\n", name)
    print(json.dumps(json.loads(path.read_text()), indent=2)[:6000])

bundle = screen.zip_directory(OUT)
print("\nBundle:", bundle)
print("Plots:", sorted(str(path) for path in (OUT / "full").glob("*.png")))

## Interpretation

A fixed-boundary numerical pass confirms only that the declared finite-difference equation was solved to the declared residual. The recorded 63% reference-condition self-heating correction is reported separately and is not substituted for the current profile, which recomputes to the bundled value. The current bundled inputs still leave the alpha self-heating, full incident-stress coolant, free-boundary 3-D MHD, orbit-resolved alpha, nuclear transport, CFD/CHT, and CHF gates unresolved or failing. Treat the generated bundle as a reproducible screening artifact, not as an engineering validation package.